# CORRIGÉ LAB 03 : Anatomie Parquet & Nettoyage Silver Partie 1 (Hygiène Structurelle)
**Cours :** Business Intelligence and Big Data Analysis (`01 31 0367 00`)  
**Public :** Enseignants & Tuteurs (Corrigé de référence exécuté — Cas 0 EuroPack Bottling)

---

## 🎯 1. Contexte & Architecture-as-Code
Corrigé de référence pour le **Cas 0 (`EuroPack Beverage Bottling`)** reliant **l'anatomie d'Apache Parquet (Cours 02)** à **l'hygiène structurelle Silver** (`bronze_staged/*.parquet` -> `silver_part1/*.parquet`). Chaque table est traitée **un fichier et un problème à la fois** (**Problème → Découverte → Solution**) ; le Contrat de Schéma Silver final et l'audit DuckDB sont traités dans le Lab 04 :

```mermaid
flowchart TD
    B0["2. Charger la couche Bronze (bronze_staged/*.parquet)"] --> P0["3. Démonstration Anatomie Parquet & Pushdown (3.1 - 3.4)"]
    P0 --> F1["4. Fichier 1 : bronze_telemetry.parquet (4.1 - 4.7)"]
    F1 --> F2["5. Fichier 2 : bronze_stoppages.parquet (5.1 - 5.2)"]
    F2 --> SC["6. Scorecard DAMA DQ Bronze & export silver_part1/*.parquet (6.1 - 6.2)"]
```

## ⚙️ 2. Configuration de l'environnement & découverte des fichiers Bronze
### 2.1 Localiser les entrées Bronze, PyArrow/DuckDB & charger les 5 tables

In [1]:
from pathlib import Path

_cwd = Path.cwd().resolve()
REPO_ROOT = None
for _cand in [_cwd, *_cwd.parents]:
    if (_cand / "03_labs_student_starters").exists():
        REPO_ROOT = _cand
        break
    for _sub in ("ife_bi_bigdata_course", "ife_bi_bigdata_student"):
        if (_cand / _sub / "03_labs_student_starters").exists():
            REPO_ROOT = _cand / _sub
            break
    if REPO_ROOT is not None:
        break
assert REPO_ROOT is not None, f"Could not locate 03_labs_student_starters from {_cwd}"

CASE_NAME = "case0_europack_bottling"
STUDENT_BRONZE_DIR = REPO_ROOT / "03_labs_student_starters" / "lab_02_bronze_ingestion_and_profiling" / "data" / CASE_NAME
REF_CASE_DATA = REPO_ROOT / "05_case_studies_reference_solutions" / CASE_NAME / "data"
BRONZE_RAW = STUDENT_BRONZE_DIR if STUDENT_BRONZE_DIR.exists() else (REF_CASE_DATA / "bronze")
SANDBOX_DATA = REPO_ROOT / "03_labs_student_starters" / "sandbox_outputs" / CASE_NAME


def resolve_stage_dir(stage_name: str) -> Path:
    """Read upstream stage from student sandbox_outputs if populated, else fallback read-only to reference solution."""
    cand = SANDBOX_DATA / stage_name
    if cand.exists() and any(cand.glob("*.parquet")):
        return cand
    ref_cand = REF_CASE_DATA / stage_name
    if ref_cand.exists():
        return ref_cand
    raise FileNotFoundError(
        f"Stage '{stage_name}' not found in {cand} (and no reference fallback exists). Run the prerequisite Lab notebook first."
    )

import duckdb
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
from IPython.display import display

BRONZE_STAGED = resolve_stage_dir("bronze_staged")
SILVER_P1 = SANDBOX_DATA / "silver_part1"
SILVER_P1.mkdir(parents=True, exist_ok=True)

df_telem = pd.read_parquet(BRONZE_STAGED / "bronze_telemetry.parquet")
df_stop = pd.read_parquet(BRONZE_STAGED / "bronze_stoppages.parquet")
df_lines = pd.read_parquet(BRONZE_STAGED / "bronze_lines.parquet")
df_shifts = pd.read_parquet(BRONZE_STAGED / "bronze_shifts.parquet")
df_formats = pd.read_parquet(BRONZE_STAGED / "bronze_bottle_formats.parquet")

print("Bronze source :", BRONZE_STAGED)
print("Silver Part 1 :", SILVER_P1)
print(
    f"Loaded 5 Bronze tables -> "
    f"telemetry: {len(df_telem):,} rows | stoppages: {len(df_stop):,} rows | "
    f"lines: {len(df_lines)} | shifts: {len(df_shifts)} | formats: {len(df_formats)}"
)


Bronze source : <REPO_ROOT>/05_case_studies_reference_solutions/case0_europack_bottling/data/bronze_staged
Silver Part 1 : <REPO_ROOT>/03_labs_student_starters/sandbox_outputs/case0_europack_bottling/silver_part1
Loaded 5 Bronze tables -> telemetry: 23,164 rows | stoppages: 292 rows | lines: 4 | shifts: 3 | formats: 3


## 📦 3. Anatomie d'Apache Parquet — En-tête, Pied de page & Pushdown (Cours 02)
Au **Cours 02 (§4.4)**, vous avez étudié pourquoi Apache Parquet surpasse CSV/JSON en analytique. Ouvrons `bronze_telemetry.parquet` comme fichier binaire brut puis avec `pyarrow.parquet.ParquetFile` — **sans charger les lignes** — pour inspecter chaque composant de sa structure interne :

```text
 [0 .. 4)              En-tête magique : b'PAR1' (4 octets)
 [4 .. debut_footer)   Row Groups -> Column Chunks -> Pages (Dictionnaire + RLE + Snappy)
 [debut_footer .. -8)  Pied de page / Footer (Thrift : schéma, offsets, stats min/max/nulls)
 [-8 .. -4)            Entier 4 octets (little-endian) = Taille du Footer en octets
 [-4 .. fin)           Pied de page magique : b'PAR1' (4 octets)
```

### 3.1 Démonstration : Octets magiques (`PAR1`), taille du Footer, métadonnées & schéma embarqué
**Pourquoi le Footer est-il écrit à la fin mais lu EN PREMIER ?** Parce qu'à l'écriture, les décalages d'octets et bornes min/max ne sont connus qu'une fois la dernière ligne traitée. À la lecture, le moteur saute directement à `EOF - 8`, lit la taille sur 4 octets, puis charge le Footer avant de décompresser la moindre page de données !

*Exécutez directement la cellule de démonstration ci-dessous pour inspecter les octets bruts et le schéma Arrow embarqué.*

In [2]:
# Démonstration prête à exécuter : inspecter la structure binaire et le footer d'un fichier Parquet
pq_path = BRONZE_STAGED / "bronze_telemetry.parquet"

# Lire le fichier en octets bruts pour vérifier l'en-tête et le pied de page binaires
raw_bytes = pq_path.read_bytes()

# 1. Vérifier la signature magique "PAR1" sur les 4 premiers et les 4 derniers octets
magic_header = raw_bytes[:4]
magic_footer = raw_bytes[-4:]

# 2. Les 4 octets juste avant le "PAR1" final (indices -8..-4) contiennent la taille du Footer
#    encodée en entier little-endian (octet de poids faible en premier)
footer_len = int.from_bytes(raw_bytes[-8:-4], "little")

# 3. Ouvrir UNIQUEMENT les métadonnées (lit le Footer sans décompresser les pages de données !)
pf = pq.ParquetFile(pq_path)

# Contrôles : signatures PAR1 valides et taille du footer identique à celle lue par PyArrow
assert magic_header == b"PAR1" and magic_footer == b"PAR1"
assert footer_len == pf.metadata.serialized_size

print(f"File size         : {len(raw_bytes):,} bytes")
print(f"Magic header/tail : {magic_header} ... {magic_footer}")
print(f"Footer length     : {footer_len:,} bytes (matches pf.metadata.serialized_size)")
print(f"Rows / Cols / RGs : {pf.metadata.num_rows:,} rows | {pf.metadata.num_columns} cols | {pf.metadata.num_row_groups} Row Group(s)")
print(f"Format / Writer   : v{pf.metadata.format_version} | {pf.metadata.created_by}")
print(f"Custom KV keys    : {list((pf.metadata.metadata or {}).keys())}")
print("\nEmbedded Arrow Schema (notice 'timestamp' & '_ingestion_timestamp_utc' are still STRING in Bronze!):")
print(pf.schema_arrow)


File size         : 261,026 bytes
Magic header/tail : b'PAR1' ... b'PAR1'
Footer length     : 6,985 bytes (matches pf.metadata.serialized_size)
Rows / Cols / RGs : 23,164 rows | 12 cols | 1 Row Group(s)
Format / Writer   : v2.6 | parquet-cpp-arrow version 26.0.1
Custom KV keys    : [b'ARROW:schema', b'pandas']

Embedded Arrow Schema (notice 'timestamp' & '_ingestion_timestamp_utc' are still STRING in Bronze!):
timestamp: string
line_id: string
bottle_format: string
ambient_temp_C: double
pasteurizer_temp_C: double
conveyor_speed_bpm: int64
filler_pressure_bar: double
valve_cycles_k: int64
bottled_units: int64
rejected_units: int64
_ingestion_timestamp_utc: string
_source_file: string
-- schema metadata --
pandas: '{"index_columns": [], "column_indexes": [], "columns": [{"name":' + 1634


### 3.2 Démonstration : Column Chunks, offsets mémoire, encodages (`Dictionnaire` + `RLE` + `Snappy`) & statistiques du Footer
Exécutez cette démonstration pour inspecter `pf.metadata.row_group(0)` colonne par colonne **uniquement depuis les métadonnées du Footer** (sans lire les pages de données) et observer :
1. **Disposition binaire en colonnes :** Chaque *Column Chunk* suit exactement le précédent (`dictionary_page_offset` / `data_page_offset`), et `4 + sum(comp_B) + footer_len + 8` égale la **taille exacte du fichier à l'octet près** !
2. **Compression Dictionnaire + RLE :** `_source_file` et `_ingestion_timestamp_utc` (`23 164` chaînes identiques) pèsent seulement ~`200–265 octets` !
3. **Audit Qualité gratuit depuis le Footer :** Les statistiques `nulls`, `min` et `max` révèlent déjà `120` valeurs nulles, `-99.0`/`85.0 bar`, `-120` bouteilles, `'LINE_A'`..`'line_c'` et `'2099-01-01'` *sans décompresser une seule ligne* !

In [3]:
# Démonstration prête à exécuter : lire l'agencement et les stats min/max/nulls depuis le Footer
rg = pf.metadata.row_group(0)

chunk_rows = []
for i in range(rg.num_columns):
    # c = descripteur de la colonne i ; st = statistiques pré-calculées (min, max, null_count)
    c = rg.column(i)
    st = c.statistics
    chunk_rows.append({
        "column": c.path_in_schema,                 # Nom de colonne dans le schéma
        "phys_type": c.physical_type,               # Type physique Parquet (INT64, DOUBLE, BYTE_ARRAY)
        "comp": c.compression,                      # Algorithme de compression (SNAPPY)
        "encodings": ",".join(c.encodings),         # Encodages (Dictionnaire + RLE + PLAIN)
        "dict_off": c.dictionary_page_offset,       # Offset en octets de la page dictionnaire
        "data_off": c.data_page_offset,             # Offset en octets des pages de données
        "raw_B": c.total_uncompressed_size,         # Taille non compressée (octets)
        "comp_B": c.total_compressed_size,          # Taille compressée sur disque (octets)
        "nulls": st.null_count if st else None,     # Nombre de valeurs manquantes (lu dans le footer)
        "min_stat": repr(st.min) if st and st.has_min_max else None,  # Valeur minimale du Row Group
        "max_stat": repr(st.max) if st and st.has_min_max else None,  # Valeur maximale du Row Group
    })

df_chunks = pd.DataFrame(chunk_rows)

# Vérifier l'équation exacte du fichier :
# 4o ("PAR1") + somme des Column Chunks compressés + Footer Thrift + 8o finaux (4o taille + 4o "PAR1")
total_payload_b = df_chunks["comp_B"].sum()
assert 4 + total_payload_b + footer_len + 8 == len(raw_bytes), "Byte map mismatch!"

print(f"Exact binary map: 4B (PAR1) + {total_payload_b:,}B (Column Chunks) + {footer_len:,}B (Footer) + 8B (len+PAR1) = {len(raw_bytes):,}B")
print("\nColumn Chunk Metadata & Footer Statistics (Row Group 0):")
print(df_chunks.to_string(index=False))


Exact binary map: 4B (PAR1) + 254,029B (Column Chunks) + 6,985B (Footer) + 8B (len+PAR1) = 261,026B

Column Chunk Metadata & Footer Statistics (Row Group 0):
                  column  phys_type   comp                encodings  dict_off  data_off  raw_B  comp_B  nulls                           min_stat                           max_stat
               timestamp BYTE_ARRAY SNAPPY PLAIN,RLE,RLE_DICTIONARY         4     25054 170373   57452      0              '2026-10-01 06:00:00'              '2099-01-01 12:00:00'
                 line_id BYTE_ARRAY SNAPPY PLAIN,RLE,RLE_DICTIONARY     57456     57529   8931    3662      0                           'LINE_A'                           'line_c'
           bottle_format BYTE_ARRAY SNAPPY PLAIN,RLE,RLE_DICTIONARY     61118     61167   5778    1714      0                        'GLASS_075'                           'PET_15'
          ambient_temp_C     DOUBLE SNAPPY PLAIN,RLE,RLE_DICTIONARY     62832     65260  33800   31598      0             

### 3.3 Démonstration : Projection Pushdown (`columns=[...]`)
Ne lire que `["line_id", "filler_pressure_bar"]` utilise les offsets du Footer pour sauter directement à ces 2 colonnes et ignorer >80 % des octets Parquet (et >96 % par rapport au CSV brut !).

In [4]:
# Démonstration prête à exécuter : Projection Pushdown (ignorer les colonnes inutiles via le Footer)
selected_cols = ["line_id", "filler_pressure_bar"]
csv_bytes = (BRONZE_RAW / "line_telemetry.csv").stat().st_size if (BRONZE_RAW / "line_telemetry.csv").exists() else 0

# PyArrow ne lit QUE les 2 colonnes demandées sur les 12 du fichier
tbl_proj = pq.read_table(pq_path, columns=selected_cols)

# Additionner uniquement la taille compressée des 2 Column Chunks ciblés (depuis le Footer)
proj_bytes = sum(
    rg.column(i).total_compressed_size
    for i in range(rg.num_columns)
    if rg.column(i).path_in_schema in selected_cols
)
print(f"Projection Pushdown ({selected_cols}):")
print(f"  Raw CSV scan: {csv_bytes:,} B -> Full Parquet chunks: {total_payload_b:,} B -> Projected 2 cols: {proj_bytes:,} B ({100 * (1 - proj_bytes / total_payload_b):.1f}% Parquet I/O skipped!)")
# Calculer le gain d'I/O réel par rapport à la lecture complète du CSV brut
if csv_bytes > 0:
    pct_csv_read = 100 * proj_bytes / csv_bytes
    print(f"  Efficiency vs Raw CSV: read only {pct_csv_read:.2f}% of CSV bytes ({100 - pct_csv_read:.2f}% I/O skipped — {csv_bytes / proj_bytes:.1f}x less data read!)")
print(f"  Projected table shape: {tbl_proj.num_rows:,} rows x {tbl_proj.num_columns} cols")


Projection Pushdown (['line_id', 'filler_pressure_bar']):
  Raw CSV scan: 1,500,439 B -> Full Parquet chunks: 254,029 B -> Projected 2 cols: 31,452 B (87.6% Parquet I/O skipped!)
  Efficiency vs Raw CSV: read only 2.10% of CSV bytes (97.90% I/O skipped — 47.7x less data read!)
  Projected table shape: 23,164 rows x 2 cols


### 3.4 Démonstration : Predicate Pushdown multi-Row-Groups (`filters=[...]`)
Sur un fichier découpé en plusieurs *Row Groups* (`row_group_size=6000`), le moteur examine les bornes `[min, max]` de chaque footer et **saute des Row Groups entiers** incompatibles ! Remarquez aussi que `'line_c'` (minuscule) se retrouve isolé dans le Row Group 3 (`'l' > 'L'` en ASCII), montrant pourquoi l'hygiène de la Section 5 est indispensable au filtrage.

In [5]:
# Démonstration prête à exécuter : Predicate Pushdown (saut de Row Groups via les stats [min, max])
# Écrire un buffer Parquet en mémoire trié par line_id et découpé en 4 Row Groups
sink = pa.BufferOutputStream()
pq.write_table(
    pa.Table.from_pandas(df_telem.sort_values("line_id"), preserve_index=False),
    sink,
    row_group_size=6000,   # Découpe les 23 164 lignes en 4 Row Groups (~6 000 lignes chacun)
    compression="snappy",
)
multi_buf = sink.getvalue()
pf_multi = pq.ParquetFile(pa.BufferReader(multi_buf))

# Lecture filtrée : PyArrow vérifie [min, max] de chaque Row Group dans le footer AVANT de décompresser
tbl_pred = pq.read_table(
    pa.BufferReader(multi_buf),
    columns=["timestamp", "line_id", "filler_pressure_bar"],
    filters=[("line_id", "=", "LINE_C")],
)

print(f"Multi-Row-Group Stats ({pf_multi.metadata.num_row_groups} Row Groups, sorted by line_id):")
for r in range(pf_multi.metadata.num_row_groups):
    rg_r = pf_multi.metadata.row_group(r)
    st_m = rg_r.column(1).statistics  # colonne 1 = line_id
    # Si "LINE_C" est hors de [min, max], le Row Group entier est ignoré sans décompression
    can_skip = not (st_m.min <= "LINE_C" <= st_m.max)
    print(f"  RG {r}: rows={rg_r.num_rows:>4} | line_id=[{st_m.min!r}, {st_m.max!r}] -> {'SKIPPED by footer!' if can_skip else 'CANDIDATE'}")
print(f"Predicate Pushdown result ('LINE_C'): {tbl_pred.num_rows:,} rows (notice uncleaned 'line_c' in RG 3 was missed!)")


Multi-Row-Group Stats (4 Row Groups, sorted by line_id):
  RG 0: rows=6000 | line_id=['LINE_A', 'LINE_B'] -> SKIPPED by footer!
  RG 1: rows=6000 | line_id=['LINE_B', 'LINE_C'] -> CANDIDATE
  RG 2: rows=6000 | line_id=['LINE_C', 'LINE_D'] -> CANDIDATE
  RG 3: rows=5164 | line_id=['LINE_D', 'line_c'] -> SKIPPED by footer!
Predicate Pushdown result ('LINE_C'): 5,106 rows (notice uncleaned 'line_c' in RG 3 was missed!)


## 🔍 4. Fichier 1 — `bronze_telemetry.parquet` (Télémesure PLC : un problème à la fois)
**Règle critique (`Immutabilité du Bronze`) :** Ne modifiez jamais les fichiers de `bronze_staged/`. Nous inspectons d'abord `df_telem` (**Découverte**), puis nous corrigeons **un problème structurel à la fois** en mémoire (**Solution**) avant de passer au Fichier 2 (`df_stop`).

### 4.1 Découverte — Clés de lignes incohérentes (`line_id`)
**Le problème :** Les journaux automates (PLC) contiennent souvent des espaces parasites (`'Line_A '`), des minuscules (`'line_c'`) ou d'anciens alias (`'LN-B'`). Avant toute correction, comparez les valeurs distinctes de `df_telem["line_id"]` avec la liste ERP officielle (`df_lines["line_id"]`).

> **Mémo syntaxe Pandas :**
> | Méthode | Rôle | Exemple |
> | :--- | :--- | :--- |
> | `Series.astype(str).str.strip().str.upper()` | Supprimer les espaces & passer en majuscules | `df_lines["line_id"].astype(str).str.strip().str.upper()` |
> | `Series.value_counts(dropna=False)` | Compter chaque valeur distincte (`NaN` inclus) | `df_telem["line_id"].value_counts(dropna=False)` |
> | `Series.isin(valeurs)` | Masque booléen : `True` si présent dans la liste | `df_telem["line_id"].isin(["Line_A ", "LN-B", "line_c"])` |

In [6]:
# Canonical ERP bottling line set (e.g. {'LINE_A', 'LINE_B', 'LINE_C', 'LINE_D'})
valid_lines = set(df_lines["line_id"].astype(str).str.strip().str.upper())
raw_line_counts = df_telem["line_id"].value_counts(dropna=False)
alias_tel_count = int(df_telem["line_id"].isin(["Line_A ", "LN-B", "line_c"]).sum())

print("Canonical ERP lines:", sorted(valid_lines))
print("\nRaw PLC line_id counts (spot 'Line_A ', 'LN-B', 'line_c' & orphan 'LINE_Z'):")
for k, cnt in raw_line_counts.items():
    flag = "  <-- CLEAN" if k in valid_lines else "  <-- DIRTY / UNKNOWN"
    print(f"  {repr(k):<14} -> {cnt:>6,} rows{flag}")
print(f"\nTotal telemetry rows needing string/alias normalization: {alias_tel_count:,}")


Canonical ERP lines: ['LINE_A', 'LINE_B', 'LINE_C', 'LINE_D']

Raw PLC line_id counts (spot 'Line_A ', 'LN-B', 'line_c' & orphan 'LINE_Z'):
  'LINE_D'       ->  5,784 rows  <-- CLEAN
  'LINE_C'       ->  5,106 rows  <-- CLEAN
  'LINE_B'       ->  4,395 rows  <-- CLEAN
  'LINE_A'       ->  4,172 rows  <-- CLEAN
  'Line_A '      ->  1,620 rows  <-- DIRTY / UNKNOWN
  'LN-B'         ->  1,390 rows  <-- DIRTY / UNKNOWN
  'line_c'       ->    692 rows  <-- DIRTY / UNKNOWN
  'LINE_Z'       ->      5 rows  <-- DIRTY / UNKNOWN

Total telemetry rows needing string/alias normalization: 3,702


### 4.2 Solution — Normaliser `line_id` en télémesure (hygiène & table d'alias)
**La correction :** Supprimez les espaces, passez en majuscules et convertissez les anciens alias (`'LN-B'` -> `'LINE_B'`) avec une fonction réutilisable `clean_line_code(val)`. Notez que la ligne fantôme `'LINE_Z'` reste **volontairement conservée** pour la Quarantaine du Lab 04 !

> **Mémo syntaxe Pandas & Python :**
> | Méthode | Rôle | Exemple |
> | :--- | :--- | :--- |
> | `dict.get(cle, defaut)` | Cherche `cle` ; renvoie `defaut` si absente | `LINE_ALIASES.get(s, s)` |
> | `Series.apply(fonction)` | Applique une fonction à chaque élément | `df_telem["line_id"] = df_telem["line_id"].apply(clean_line_code)` |
> | `Series.unique()` | Liste les valeurs distinctes pour vérification | `sorted(df_telem["line_id"].unique())` |

In [7]:
# Canonical alias lookup table mapping legacy variants -> standard ERP bottling line IDs
LINE_ALIASES = {
    "LN-A": "LINE_A", "LINE-A": "LINE_A", "L_A": "LINE_A",
    "LN-B": "LINE_B", "LINE-B": "LINE_B", "L_B": "LINE_B",
    "LN-C": "LINE_C", "LINE-C": "LINE_C", "L_C": "LINE_C",
    "LN-D": "LINE_D", "LINE-D": "LINE_D", "L_D": "LINE_D",
}

def clean_line_code(val):
    if pd.isna(val):                              # Guard against missing/NaN values
        return "UNKNOWN"
    s = str(val).strip().upper()                  # Remove spaces ('Line_A ') & standardize case ('line_c')
    return LINE_ALIASES.get(s, s)                 # Map alias ('LN-B' -> 'LINE_B') or keep as-is ('LINE_Z')

print("BEFORE -> telemetry line_id:", sorted(df_telem["line_id"].dropna().unique()))

df_lines["line_id"] = df_lines["line_id"].apply(clean_line_code)
df_telem["line_id"] = df_telem["line_id"].apply(clean_line_code)

print("AFTER  -> telemetry line_id:", sorted(df_telem["line_id"].dropna().unique()))
assert df_telem["line_id"].str.contains(" ").sum() == 0, "Whitespace still present in line_id!" 


BEFORE -> telemetry line_id: ['LINE_A', 'LINE_B', 'LINE_C', 'LINE_D', 'LINE_Z', 'LN-B', 'Line_A ', 'line_c']
AFTER  -> telemetry line_id: ['LINE_A', 'LINE_B', 'LINE_C', 'LINE_D', 'LINE_Z']


### 4.3 Découverte — Pourquoi les horodatages en texte cassent les pipelines (`timestamp`)
**Le problème :** En Bronze, `df_telem["timestamp"]` a volontairement été conservé en texte brut (`string` / `object`). Avant de le convertir, exécutez trois expériences montrant pourquoi des dates en texte cassent un pipeline :
1. **Arithmétique impossible :** Soustraire `pd.Timedelta(hours=6)` lève une `TypeError` (et `.interpolate(method='time')` du Lab 04 échoue).
2. **Tri silencieux faux :** Les chaînes sont comparées caractère par caractère en ASCII (`'09/10/2026' > '01/11/2026'` vaut `True` ; `'2026-10-01T06:00:00' > '2026-10-01 07:00:00'` vaut `True` car `'T' > ' '` !).
3. **~8,5× plus de RAM :** Le type natif `datetime64[ns]` stocke 8 octets par ligne au lieu d'objets chaînes Python.

> **Mémo syntaxe Pandas :**
> | Méthode | Rôle | Exemple |
> | :--- | :--- | :--- |
> | `Series.memory_usage(deep=True)` | Empreinte RAM exacte en octets (chaînes incluses) | `df_telem["timestamp"].memory_usage(deep=True)` |
> | `pd.to_datetime(s, errors="coerce")` | Convertir texte -> `datetime64[ns]` (invalide -> `NaT`) | `pd.to_datetime(df_telem["timestamp"], errors="coerce")` |
> | `pd.Timedelta(hours=6)` | Durée pour l'arithmétique temporelle | `ts - pd.Timedelta(hours=6)` |

In [8]:
# Experiment 1: Temporal arithmetic fails on raw string timestamps
try:
    _ = df_telem["timestamp"].head(3) - pd.Timedelta(hours=6)
except TypeError as err:
    print(f"[Experiment 1] String timestamp math FAILED as expected -> TypeError: {err}")

# Experiment 2: Silent alphabetical vs chronological ordering bugs
eu_cmp_str = "09/10/2026" > "01/11/2026"
iso_cmp_str = "2026-10-01T06:00:00" > "2026-10-01 07:00:00"  # 'T' (ASCII 84) > ' ' (ASCII 32)!
num_cmp_str = "9" > "10"
print(f"[Experiment 2] String traps -> '09/10/2026' > '01/11/2026' is {eu_cmp_str}, '2026-10-01T06:00:00' > '2026-10-01 07:00:00' is {iso_cmp_str}, '9' > '10' is {num_cmp_str}!")

# Experiment 3: RAM usage comparison (object/str vs native datetime64[ns])
ram_str_bytes = df_telem["timestamp"].memory_usage(deep=True)
ram_dt_bytes = pd.to_datetime(df_telem["timestamp"], errors="coerce").memory_usage(deep=True)
print(f"[Experiment 3] RAM for 'timestamp' -> object/str: {ram_str_bytes:,} B vs datetime64[ns]: {ram_dt_bytes:,} B ({ram_str_bytes / ram_dt_bytes:.2f}x smaller!)")


[Experiment 1] String timestamp math FAILED as expected -> TypeError: unsupported operand type(s) for -: 'str' and 'Timedelta'
[Experiment 2] String traps -> '09/10/2026' > '01/11/2026' is True, '2026-10-01T06:00:00' > '2026-10-01 07:00:00' is True, '9' > '10' is True!
[Experiment 3] RAM for 'timestamp' -> object/str: 1,575,284 B vs datetime64[ns]: 185,444 B (8.49x smaller!)


### 4.4 Solution — Convertir `timestamp` & gérer la bascule de nuit `S3`
**La correction :** Convertissez `df_telem["timestamp"]` en `datetime64[ns]` natif. Attribuez ensuite l'équipe (`S1` : `06–14`, `S2` : `14–22`, `S3` : `22–06`) et calculez **`operational_shift_date = (ts - 6 heures).dt.date`** pour que les relevés entre `00:00` et `05:45` restent rattachés à la date de début de l'équipe de nuit — en vrai type **`datetime.date`** (`DATE` / `date32[day]`, **sans** `.astype(str)` !).

> **Mémo syntaxe Pandas :**
> | Méthode | Rôle | Exemple |
> | :--- | :--- | :--- |
> | `pd.to_datetime(s, errors="coerce")` | Convertir une colonne texte en `datetime64[ns]` | `df_telem["timestamp"] = pd.to_datetime(df_telem["timestamp"], errors="coerce")` |
> | `Series.dt.hour` | Extraire l'heure `0..23` d'une Series datetime | `ts.dt.hour.fillna(-1).astype(int)` |
> | `(ts - pd.Timedelta(hours=6)).dt.date` | Reculer de 6h et extraire une `DATE` native | `(df_telem["timestamp"] - pd.Timedelta(hours=6)).dt.date` |

In [9]:
# Reusable helper: map each timestamp's hour to S1 (06-14), S2 (14-22), or S3 (22-06)
def assign_shift_code(ts: pd.Series) -> pd.Series:
    return pd.Series(
        ["S1" if 6 <= h < 14 else ("S2" if 14 <= h < 22 else "S3") for h in ts.dt.hour.fillna(-1).astype(int)],
        index=ts.index,
    )

# 1. Parse PLC telemetry timestamps from text into native datetime64[ns]
df_telem["timestamp"] = pd.to_datetime(df_telem["timestamp"], errors="coerce")

# 2. Assign 8-hour shift_code and operational_shift_date (rolling 00:00-05:59 back to previous day)
df_telem["shift_code"] = assign_shift_code(df_telem["timestamp"])
df_telem["operational_shift_date"] = (df_telem["timestamp"] - pd.Timedelta(hours=6)).dt.date

print("Telemetry timestamp dtype:", df_telem["timestamp"].dtype)
sample_s3 = df_telem[df_telem["timestamp"].dt.hour == 2][["timestamp", "shift_code", "operational_shift_date"]].head(3)
print("\nSample S3 post-midnight rollover check (02:00 belongs to previous operational date):")
print(sample_s3.to_string(index=False))


Telemetry timestamp dtype: datetime64[ns]

Sample S3 post-midnight rollover check (02:00 belongs to previous operational date):
          timestamp shift_code operational_shift_date
2026-10-02 02:00:00         S3             2026-10-01
2026-10-02 02:00:00         S3             2026-10-01
2026-10-02 02:00:00         S3             2026-10-01


### 4.5 Découverte — Doublons MQTT en télémesure
**Le problème :** Les passerelles MQTT renvoient parfois deux fois le même message en fin de fichier pour la même clé `(timestamp, line_id, bottle_format)`. Inspecter les doublons **après** la normalisation de `line_id` évite d'en manquer — et **trier par `KEY_COLS` avant `.head(4)`** place chaque ligne d'origine juste à côté de sa copie (en conservant l'index pour voir la ligne initiale vs la copie ajoutée en fin de fichier !).

> **Mémo syntaxe Pandas :**
> | Méthode | Rôle | Exemple |
> | :--- | :--- | :--- |
> | `df.duplicated(subset=KEY_COLS)` | Masque booléen des doublons (hors 1re occurrence) | `df_telem.duplicated(subset=KEY_COLS).sum()` |
> | `df.duplicated(subset=KEY_COLS, keep=False)` | Marque **toutes** les copies de chaque clé en double | `df_telem.loc[df_telem.duplicated(subset=KEY_COLS, keep=False)]` |
> | `.sort_values(KEY_COLS)` | Regrouper les paires identiques côte à côte avant `.head()` | `all_dupes.sort_values(KEY_COLS).groupby(KEY_COLS, sort=False).head(2).head(4)` |

In [10]:
KEY_COLS = ["timestamp", "line_id", "bottle_format"]

# Discovery (Read-Only): Count & display actual duplicate pairs side-by-side with original row indices
dupes_count = int(df_telem.duplicated(subset=KEY_COLS).sum())
print(f"[Discovery] Found {dupes_count:,} duplicate telemetry rows on {KEY_COLS}.")

all_dupes = df_telem.loc[df_telem.duplicated(subset=KEY_COLS, keep=False), [*KEY_COLS, "bottled_units"]]
sample_dupes = all_dupes.sort_values(KEY_COLS).groupby(KEY_COLS, sort=False).head(2).head(4)
print("\nSample duplicate pairs (leftmost column = original file row index):")
print(sample_dupes.to_string())


[Discovery] Found 114 duplicate telemetry rows on ['timestamp', 'line_id', 'bottle_format'].

Sample duplicate pairs (leftmost column = original file row index):
                timestamp line_id bottle_format  bottled_units
10    2026-10-01 06:30:00  LINE_C        PET_15            934
23081 2026-10-01 06:30:00  LINE_C        PET_15            934
17    2026-10-01 07:00:00  LINE_B     GLASS_075            809
23121 2026-10-01 07:00:00  LINE_B     GLASS_075            809


### 4.6 Solution — Dédupliquer la télémesure (`keep='last'`)
**La correction :** Supprimez les doublons sur `KEY_COLS` en gardant la dernière réception (`keep="last"` suit l'ordre du fichier, conservant ainsi la ligne `23081` plutôt que `10`), puis appelez `.copy()` pour éviter tout `SettingWithCopyWarning`.

> **Mémo syntaxe Pandas :**
> | Méthode | Rôle | Exemple |
> | :--- | :--- | :--- |
> | `df.drop_duplicates(subset=KEY_COLS, keep="last").copy()` | Garder la dernière occurrence & renvoyer une copie propre | `df_telem = df_telem.drop_duplicates(subset=KEY_COLS, keep="last").copy()` |

In [11]:
before_tel = len(df_telem)
df_telem = df_telem.drop_duplicates(subset=KEY_COLS, keep="last").copy()
remaining = int(df_telem.duplicated(subset=KEY_COLS).sum())
assert remaining == 0, "Duplicates still remain in df_telem!"
print(f"[Solution] Removed {before_tel - len(df_telem):,} duplicates -> kept {len(df_telem):,} rows (remaining duplicates: {remaining}).")


[Solution] Removed 114 duplicates -> kept 23,050 rows (remaining duplicates: 0).


### 4.7 Découverte — Anomalies télémesure réservées au Lab 04 (lecture seule)
**Le problème :** L'hygiène structurelle de `df_telem` est terminée, **mais des valeurs capteurs corrompues subsistent** (pressions impossibles `-99.0`/`85.0 bar`, `pasteurizer_temp_C` manquant, dates futures `2099`, `rejected_units > bottled_units`, ligne orpheline `LINE_Z`). Ne les supprimez **PAS** ici — comptez-les pour le sas de Quarantaine et l'Interpolation du Lab 04 !

> **Mémo syntaxe Pandas :**
> | Méthode / Opérateur | Rôle | Exemple |
> | :--- | :--- | :--- |
> | `(cond_a) \| (cond_b)` | Combiner deux masques avec OU bit-à-bit (parenthèses obligatoires !) | `(df_telem["filler_pressure_bar"] < 0) \| (df_telem["filler_pressure_bar"] > 12)` |
> | `Series.isna().sum()` | Compter les valeurs manquantes (`NaN` / `NaT`) | `df_telem["pasteurizer_temp_C"].isna().sum()` |
> | `~Series.isin(valid_lines)` | Inverser un masque pour repérer les clés orphelines | `(~df_telem["line_id"].isin(valid_lines)).sum()` |

In [12]:
bad_pressure_count = int(((df_telem["filler_pressure_bar"] < 0) | (df_telem["filler_pressure_bar"] > 12)).sum())
null_temp_count = int(df_telem["pasteurizer_temp_C"].isna().sum())
orphan_tel_count = int((~df_telem["line_id"].isin(valid_lines)).sum())
future_tel_count = int((df_telem["timestamp"] > "2027-01-01").sum())
bad_reject_count = int(
    ((df_telem["bottled_units"] < 0) | (df_telem["rejected_units"] < 0) | (df_telem["rejected_units"] > df_telem["bottled_units"])).sum()
)

print("=== FILE 1 (df_telem) READ-ONLY ANOMALIES FOR LAB 04 ===")
print(f"  Orphan line_id (LINE_Z)               : {orphan_tel_count}")
print(f"  Future timestamps (> 2027-01-01)      : {future_tel_count}")
print(f"  Out-of-bounds pressure (<0 or >12 bar): {bad_pressure_count}")
print(f"  Impossible unit counts                : {bad_reject_count}")
print(f"  Missing pasteurizer_temp_C (NaN)      : {null_temp_count}  (to interpolate in Lab 04, NOT drop!)")


=== FILE 1 (df_telem) READ-ONLY ANOMALIES FOR LAB 04 ===
  Orphan line_id (LINE_Z)               : 5
  Future timestamps (> 2027-01-01)      : 1
  Out-of-bounds pressure (<0 or >12 bar): 40
  Impossible unit counts                : 4
  Missing pasteurizer_temp_C (NaN)      : 120  (to interpolate in Lab 04, NOT drop!)


## 📋 5. Fichier 2 — `bronze_stoppages.parquet` (Journaux d'arrêts GMAO : Découverte → Solution)
Le Fichier 1 (`df_telem`) étant entièrement traité, appliquons exactement la même démarche **Découverte → Solution** au Fichier 2 (`df_stop`, 292 journaux d'arrêts GMAO) de manière isolée.

### 5.1 Découverte — Clés, dérive de casse, doublons & journal fantôme
**Le problème :** Auditez `df_stop` pour quatre anomalies précises :
1. Codes `line_code` sales (`'Line_A '`, `'LN-B'`) et ligne orpheline `'LINE_Z'`.
2. Dérive de casse dans `stoppage_type` (`'unplanned'` vs `'Unplanned'` vs `'Preventive'`).
3. Doublons sur `log_id`.
4. Journal fantôme futur `STOP-20269999` (`2099-01-01`, `LINE_Z`) — à repérer ici et mettre en quarantaine au Lab 04.

> **Mémo syntaxe Pandas :**
> | Méthode | Rôle | Exemple |
> | :--- | :--- | :--- |
> | `Series.value_counts(dropna=False)` | Compter chaque catégorie distincte | `df_stop["stoppage_type"].value_counts(dropna=False)` |
> | `df.duplicated(subset=["log_id"]).sum()` | Compter les doublons d'identifiant | `df_stop.duplicated(subset=["log_id"]).sum()` |
> | `df.loc[mask, cols]` | Afficher uniquement les lignes signalées | `df_stop.loc[stop_bad_mask, ["log_id", "event_date", "line_code"]]` |

In [13]:
stop_type_counts = df_stop["stoppage_type"].value_counts(dropna=False)
stop_dupes_count = int(df_stop.duplicated(subset=["log_id"]).sum())

clean_stop_codes = df_stop["line_code"].astype(str).str.strip().str.upper()
stop_ts = pd.to_datetime(df_stop["event_date"], errors="coerce")
is_future_or_invalid = stop_ts.isna() | (stop_ts > "2027-01-01")
stop_future_orphan_count = int((stop_ts > "2027-01-01").sum())
stop_unplanned_lower_count = int((df_stop["stoppage_type"] == "unplanned").sum())

print("1. Raw stoppage line_code values:", sorted(df_stop["line_code"].dropna().unique()))
print("\n2. Raw stoppage_type counts (notice 'unplanned' vs 'Unplanned'):")
print(stop_type_counts.to_string())
print(f"\n3. Duplicate log_id rows: {stop_dupes_count} | Defensive downtime_min <= 0: {(df_stop['downtime_min'] <= 0).sum()}")
print("\n4. Phantom / future stoppage log (to keep for Lab 04 Quarantine):")
print(df_stop.loc[is_future_or_invalid | (clean_stop_codes == "LINE_Z"), ["log_id", "event_date", "line_code", "stoppage_type", "root_cause", "downtime_min"]].to_string(index=False))


1. Raw stoppage line_code values: ['LINE_A', 'LINE_B', 'LINE_C', 'LINE_D', 'LINE_Z', 'LN-B', 'Line_A ']

2. Raw stoppage_type counts (notice 'unplanned' vs 'Unplanned'):
stoppage_type
Unplanned     177
Preventive     79
unplanned      36

3. Duplicate log_id rows: 2 | Defensive downtime_min <= 0: 0

4. Phantom / future stoppage log (to keep for Lab 04 Quarantine):
       log_id          event_date line_code stoppage_type  root_cause  downtime_min
STOP-20269999 2099-01-01 03:00:00    LINE_Z     Unplanned Phantom Log            60


### 5.2 Solution — Nettoyer `df_stop` (clés, catégories, `event_date`, doublons & date `S3`)
**La correction :** Réutilisez `clean_line_code` et `assign_shift_code` de la Section 4, uniformisez `stoppage_type` (`'Unplanned'`, `'Preventive'`, `'Corrective'`), convertissez `event_date` en `datetime64[ns]`, dédupliquez sur `log_id` (`keep='last'`) et calculez `shift_code` + `operational_shift_date`.

> **Mémo syntaxe Pandas :**
> | Méthode | Rôle | Exemple |
> | :--- | :--- | :--- |
> | `Series.apply(fonction)` | Standardiser les textes élément par élément | `df_stop["stoppage_type"] = df_stop["stoppage_type"].apply(standardize_stoppage_type)` |
> | `pd.to_datetime(s, errors="coerce")` | Convertir `event_date` en `datetime64[ns]` | `df_stop["event_date"] = pd.to_datetime(df_stop["event_date"], errors="coerce")` |
> | `df.drop_duplicates(subset=["log_id"], keep="last").copy()` | Supprimer les journaux en double proprement | `df_stop = df_stop.drop_duplicates(subset=["log_id"], keep="last").copy()` |

In [14]:
def standardize_stoppage_type(val):
    s = str(val).strip().lower()
    if "prevent" in s or "pm" in s:
        return "Preventive"
    if "correct" in s:
        return "Corrective"
    return "Unplanned"

# 1. Normalize line_code (reusing clean_line_code) and stoppage_type
df_stop["line_code"] = df_stop["line_code"].apply(clean_line_code)
df_stop["stoppage_type"] = df_stop["stoppage_type"].apply(standardize_stoppage_type)

# 2. Parse event_date into native datetime64[ns]
df_stop["event_date"] = pd.to_datetime(df_stop["event_date"], errors="coerce")

# 3. Deduplicate on log_id (keep='last' + .copy())
before_stop = len(df_stop)
df_stop = df_stop.drop_duplicates(subset=["log_id"], keep="last").copy()

# 4. Assign shift_code & operational_shift_date (reusing assign_shift_code)
df_stop["shift_code"] = assign_shift_code(df_stop["event_date"])
df_stop["operational_shift_date"] = (df_stop["event_date"] - pd.Timedelta(hours=6)).dt.date

print("Clean stoppage line_code    :", sorted(df_stop["line_code"].unique()))
print("Clean stoppage_type counts  :", df_stop["stoppage_type"].value_counts().to_dict())
print(f"Stoppage dedup removed {before_stop - len(df_stop):,} duplicate logs (kept {len(df_stop):,}; STOP-20269999 preserved for Lab 04).")


Clean stoppage line_code    : ['LINE_A', 'LINE_B', 'LINE_C', 'LINE_D', 'LINE_Z']
Clean stoppage_type counts  : {'Unplanned': 212, 'Preventive': 78}
Stoppage dedup removed 2 duplicate logs (kept 290; STOP-20269999 preserved for Lab 04).


## 📊 6. Triage consolidé — Scorecard DAMA DQ Bronze & export intermédiaire
### 6.1 Compiler la Scorecard DAMA DQ Bronze
**L'objectif :** Rassemblez les constats du Fichier 1 (`df_telem`, Section 4) et du Fichier 2 (`df_stop`, Section 5) dans une table d'audit. Notez la distinction clé pour le Lab 04 :
- **Erreurs de ligne irrécupérables** (`DQR-101, 102, 104, 105` — 11 lignes) → **Quarantaine & retrait de la ligne**.
- **Glitches capteur isolés sur des lignes de production valides** (`DQR-103` pics de pression + `DQR-TEMP` températures nulles) → **Journaliser, mettre à `NaN` & interpoler dans le temps** pour ne perdre aucune bouteille produite !

> **Mémo syntaxe Pandas :**
> | Méthode | Rôle | Exemple |
> | :--- | :--- | :--- |
> | `pd.DataFrame([{...}, ...])` | Construire un tableau structuré depuis une liste de dicts | `pd.DataFrame([{"Rule_ID": "DQR-101", "Count": 5}])` |

In [15]:
dq_scorecard = pd.DataFrame([
    {"Rule_ID": "DQR-KEY",  "Source": "PLC + CMMS", "Anomaly": "Whitespace/casing/alias drift ('Line_A ', 'LN-B', 'unplanned')", "Count": alias_tel_count + stop_unplanned_lower_count, "Planned_Action": "Lab 03 Done — Normalized"},
    {"Rule_ID": "DQR-107",  "Source": "PLC + CMMS", "Anomaly": "Duplicate rows (114 telemetry + 2 stoppage logs)",              "Count": dupes_count + stop_dupes_count,               "Planned_Action": "Lab 03 Done — Deduplicated (keep='last')"},
    {"Rule_ID": "DQR-101",  "Source": "PLC CSV",    "Anomaly": "Orphan ghost asset (LINE_Z)",                                   "Count": orphan_tel_count,                             "Planned_Action": "Lab 04 Quarantine Row (BLOCKER)"},
    {"Rule_ID": "DQR-102",  "Source": "PLC CSV",    "Anomaly": "Future 2099-* telemetry timestamp",                             "Count": future_tel_count,                             "Planned_Action": "Lab 04 Quarantine Row"},
    {"Rule_ID": "DQR-104",  "Source": "PLC CSV",    "Anomaly": "Negative or rejected_units > bottled_units",                    "Count": bad_reject_count,                             "Planned_Action": "Lab 04 Quarantine Row"},
    {"Rule_ID": "DQR-105",  "Source": "CMMS JSON",  "Anomaly": "Phantom/future stoppage log (STOP-20269999)",                   "Count": stop_future_orphan_count,                     "Planned_Action": "Lab 04 Quarantine Row"},
    {"Rule_ID": "DQR-103",  "Source": "PLC CSV",    "Anomaly": "Impossible filler_pressure_bar (-99.0 / 85.0 bar)",             "Count": bad_pressure_count,                           "Planned_Action": "Lab 04 Log, Nullify & Time Interpolate (Keep Bottles!)"},
    {"Rule_ID": "DQR-TEMP", "Source": "PLC CSV",    "Anomaly": "Missing pasteurizer_temp_C (NaN)",                              "Count": null_temp_count,                              "Planned_Action": "Lab 04 Time Interpolate (NO dropna!)"},
])
print("=== BRONZE DATA QUALITY SCORECARD (TRIAGE PLAN) ===")
print(dq_scorecard.to_string(index=False))


=== BRONZE DATA QUALITY SCORECARD (TRIAGE PLAN) ===
 Rule_ID     Source                                                        Anomaly  Count                                         Planned_Action
 DQR-KEY PLC + CMMS Whitespace/casing/alias drift ('Line_A ', 'LN-B', 'unplanned')   3738                               Lab 03 Done — Normalized
 DQR-107 PLC + CMMS               Duplicate rows (114 telemetry + 2 stoppage logs)    116               Lab 03 Done — Deduplicated (keep='last')
 DQR-101    PLC CSV                                    Orphan ghost asset (LINE_Z)      5                        Lab 04 Quarantine Row (BLOCKER)
 DQR-102    PLC CSV                              Future 2099-* telemetry timestamp      1                                  Lab 04 Quarantine Row
 DQR-104    PLC CSV                     Negative or rejected_units > bottled_units      4                                  Lab 04 Quarantine Row
 DQR-105  CMMS JSON                    Phantom/future stoppage log (STOP-20269

### 6.2 Sauvegarder les tables intermédiaires dans `silver_part1/*.parquet` (Passage au Lab 04)
**L'objectif :** Exportez nos 5 tables structurellement nettoyées dans `silver_part1/*.parquet` afin que le **Lab 04** puisse enchaîner sur les valeurs aberrantes (`IQR` vs `3-Sigma`), la Quarantaine & réparation capteur, l'Interpolation temporelle et le **Contrat de Schéma Silver final + audit DuckDB**.

> **Mémo syntaxe Pandas :**
> | Méthode | Rôle | Exemple |
> | :--- | :--- | :--- |
> | `df.to_parquet(chemin, index=False, compression="snappy")` | Sauvegarder en Parquet Snappy sans fuite `__index_level_0__` | `df_out.to_parquet(SILVER_P1 / fname, index=False, compression="snappy")` |

In [16]:
assert df_telem["line_id"].str.contains(" ").sum() == 0, "Whitespace remains in line_id!"
assert set(df_stop["stoppage_type"]).issubset({"Unplanned", "Preventive", "Corrective"}), "Unstandardized stoppage types!"

outputs = {
    "silver_part1_telemetry.parquet": df_telem,
    "silver_part1_stoppages.parquet": df_stop,
    "silver_lines.parquet": df_lines,
    "silver_shifts.parquet": df_shifts,
    "silver_bottle_formats.parquet": df_formats,
}
for fname, df_out in outputs.items():
    df_out.to_parquet(SILVER_P1 / fname, index=False, compression="snappy")
    print(f"  [SAVED] {fname:<34} ({len(df_out):>6,} rows)")

print(f"\nIntermediate Silver Part 1 Parquets ready in: {SILVER_P1}")
print("NOTE: LINE_Z, 2099 future dates, -99/85 bar sensor spikes & NaN temps are intentionally preserved for Lab 04!")


  [SAVED] silver_part1_telemetry.parquet     (23,050 rows)
  [SAVED] silver_part1_stoppages.parquet     (   290 rows)
  [SAVED] silver_lines.parquet               (     4 rows)
  [SAVED] silver_shifts.parquet              (     3 rows)
  [SAVED] silver_bottle_formats.parquet      (     3 rows)

Intermediate Silver Part 1 Parquets ready in: <REPO_ROOT>/03_labs_student_starters/sandbox_outputs/case0_europack_bottling/silver_part1
NOTE: LINE_Z, 2099 future dates, -99/85 bar sensor spikes & NaN temps are intentionally preserved for Lab 04!
